# SOXLBacktest qlib reference v1

Executable reference run for the cleaned qlib architecture. The notebook validates the migrated SOXLBacktest reference logic and writes all sidecars into this card.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Could not find repo root')
    ROOT = ROOT.parent
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'soxlbacktest_qlib_reference_v1'
CARD.mkdir(parents=True, exist_ok=True)
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

PosixPath('/Users/rziiiii/dev/workspace/.worktrees/cleanup-soxl-qlib-v1')

In [2]:
from soxl_backtest.framework import ExperimentSpec, QlibBacktestRunner, load_qlib_frame
from soxl_backtest.reference import replay_soxlbacktest_reference

qlib_frame = load_qlib_frame(start='2026-01-01', end='2026-06-09')
qlib_audit = {
    'rows': int(len(qlib_frame)),
    'start': qlib_frame['date'].min().date().isoformat(),
    'end': qlib_frame['date'].max().date().isoformat(),
    'data_source': qlib_frame.attrs['data_source'],
    'qlib_calendar_path': str(Path(qlib_frame.attrs['qlib_calendar_path']).relative_to(ROOT)),
}
qlib_audit

{'rows': 109,
 'start': '2026-01-02',
 'end': '2026-06-09',
 'data_source': 'qlib_calendar_plus_canonical_panel',
 'qlib_calendar_path': 'data/qlib/soxl/calendars/day.txt'}

In [3]:
spec = ExperimentSpec(
    experiment='soxlbacktest_qlib_reference_v1',
    direction='model_selection',
    start='2026-01-01',
    end='2026-06-09',
    changed_layer='selection',
    role='replace',
    baseline='legacy_soxl_backtest',
    comparison_key='soxlbacktest_qlib_reference_v1:ytd_2026',
)
ytd = QlibBacktestRunner(spec).run_reference(CARD)
june = replay_soxlbacktest_reference(start='2026-06-01', end='2026-06-09')
window_metrics = pd.DataFrame([
    {'window': 'ytd_2026', **ytd.metrics},
    {'window': 'june_2026', **june.metrics},
])
window_metrics.to_csv(CARD / 'window_metrics.csv', index=False)
combined_daily = pd.concat([
    ytd.daily.assign(window='ytd_2026'),
    june.daily.assign(window='june_2026'),
], ignore_index=True)
combined_trades = pd.concat([
    ytd.trades.assign(window='ytd_2026'),
    june.trades.assign(window='june_2026'),
], ignore_index=True)
combined_daily.to_csv(CARD / 'combined_daily.csv', index=False)
combined_trades.to_csv(CARD / 'combined_trades.csv', index=False)
window_metrics[['window', 'rulebase_return_pct', 'rulebase_max_drawdown_pct', 'end_position_pct', 'trade_count']]

[local] 读取 130 条数据
[local] 读取 26 条数据


,window,rulebase_return_pct,rulebase_max_drawdown_pct,end_position_pct,trade_count
0,ytd_2026,132.077978,13.554089,15.91,3
1,june_2026,3.138960,1.528209,29.94,1


In [4]:
summary = {
    'experiment': 'soxlbacktest_qlib_reference_v1',
    'direction': 'model_selection',
    'decision': 'reference_parity_recorded',
    'qlib_audit': qlib_audit,
    'windows': {
        'ytd_2026': ytd.metrics,
        'june_2026': june.metrics,
    },
    'acceptance': {
        'ytd_return_pct': 132.07797795135497,
        'june_return_pct': 3.1389599999999973,
        'trade_dates': ['2026-03-30', '2026-05-27', '2026-06-05'],
    },
    'artifacts': {
        'notebook': 'soxlbacktest_qlib_reference_v1.ipynb',
        'daily': 'daily.csv',
        'trades': 'trades.csv',
        'metrics': 'metrics.json',
        'experiment_spec': 'experiment_spec.json',
        'window_metrics': 'window_metrics.csv',
        'combined_daily': 'combined_daily.csv',
        'combined_trades': 'combined_trades.csv',
        'summary': 'summary.json',
    },
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary

{'experiment': 'soxlbacktest_qlib_reference_v1',
 'direction': 'model_selection',
 'decision': 'reference_parity_recorded',
 'qlib_audit': {'rows': 109,
  'start': '2026-01-02',
  'end': '2026-06-09',
  'data_source': 'qlib_calendar_plus_canonical_panel',
  'qlib_calendar_path': 'data/qlib/soxl/calendars/day.txt'},
 'windows': {'ytd_2026': {'requested_start': '2026-01-01',
   'start': '2026-01-02',
   'end': '2026-06-09',
   'rows': 109,
   'initial_capital': 100000.0,
   'final_capital': 232077.97795135496,
   'soxl_buyhold_return_pct': 326.9263184466205,
   'rulebase_return_pct': 132.07797795135497,
   'rulebase_max_drawdown_pct': 13.554088542803521,
   'average_position_pct': 16.220733944954127,
   'end_position_pct': 15.909999999999998,
   'trade_count': 3},
  'june_2026': {'requested_start': '2026-06-01',
   'start': '2026-06-01',
   'end': '2026-06-09',
   'rows': 7,
   'initial_capital': 100000.0,
   'final_capital': 103138.96,
   'soxl_buyhold_return_pct': -11.165925208122275,


## Conclusion

The qlib-backed reference workflow reproduces the legacy SOXLBacktest behavior on the required windows. The 2026 YTD run keeps the known `132.07797795135497%` rulebase return with three trades, and the June fresh-start run keeps the known `3.13896%` return with one trade. This card is the active reference baseline for future single-factor, multi-factor, and model-selection notebooks.